# Chapter 7: Gromov-Witten Invariants

    **Source orientation:** McDuff-Salamon, *J-holomorphic Curves and Symplectic Topology*, Chapter 7, printed pp. 201-256; PDF pp. 216-271. Sections: 7.1-7.5.

    ## Chapter Goal

    Counting pseudoholomorphic spheres, variations on the definition, graph counts, rational curves in projective spaces, and Gromov-Witten axioms.

    The guiding question is:

    > How does a compact, regular, constrained moduli problem become a deformation-invariant number?

    ## Computational Translation Guide

    | Chapter language | Computational object | Inspection target / check |
| --- | --- | --- |
| GW invariant | signed count of constrained curves | dimension zero |
| incidence constraint | cycle pulled back by evaluation | codimension match |
| projective-space example | degree d rational curves | 3d-1 point count |
| axiom | identity among counts | consistency check |
| deformation invariance | pseudocycle cobordism | unchanged count |


## Standalone Reading Guide

    This is the counting chapter. The previous machinery is assembled into invariants by intersecting evaluation pseudocycles with constraints of matching codimension. If the constrained moduli space is zero-dimensional, compact in the relevant sense, and oriented, the signed count is stable under deformation. The emphasis is not just that curves are counted, but that the count survives the choices used to define it.

Projective space supplies the most concrete mental model. In CP2, degree d rational curves are expected to be isolated after imposing 3d-1 generic point conditions. Lines through two points and conics through five points are familiar classical cases; cubics already show the richer enumerative behavior that Gromov-Witten theory organizes. The axioms give algebraic relations among such counts and prepare for the quantum product.

The notebook uses a small table of standard low-degree CP2 counts as anchors. The plotted dimension ledger explains why those are counts rather than families. The graph artifact records the proof dependencies: regularity gives a smooth moduli space, compactness identifies acceptable limits, evaluation maps impose constraints, and cobordism arguments protect the final number.

    ## Topics In This Notebook

    - zero-dimensional constrained moduli spaces
- variations of marked points, classes, and graph counts
- rational curves in projective space as a guiding example
- Kontsevich-Manin style axioms as computational structure
- splitting and deformation ideas that motivate later gluing

    ## Visualization Storyboard

    - A projective-plane ledger plots the constraint number 3d-1 against known low-degree counts.
- A dependency graph shows how regularity, compactness, and evaluation meet in the invariant.
- A table records dimension-zero checks for degrees one through four.


In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "J-Holomorphic-Curves-and-Symplectic-Topology"]:
            if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
                return candidate
    raise RuntimeError("JHCST book root not found")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib

UNIT = 'chapter-07'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / UNIT
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
HTML_DIR = ARTIFACT_ROOT / "html"
for folder in [FIG_DIR, CHECK_DIR, TABLE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BOOK_ROOT


## Proof Visual: Dependency Map

The graph below is a compact proof-state diagram. Read an arrow as "this idea must be under control before the next one can be used." The point is not to replace the analysis with a graph, but to keep the logical dependencies inspectable while the chapter moves between local equations, moduli spaces, compactness, and algebraic counts.


In [ ]:
CONCEPT_NODES = ['GW invariant', 'incidence constraint', 'projective-space example', 'axiom', 'deformation invariance']
CONCEPT_EDGES = [('GW invariant', 'incidence constraint'), ('incidence constraint', 'projective-space example'), ('projective-space example', 'axiom'), ('axiom', 'deformation invariance')]

G = nx.DiGraph()
G.add_nodes_from(CONCEPT_NODES)
G.add_edges_from(CONCEPT_EDGES)
pos = nx.spring_layout(G, seed=34, k=1.35)

fig, ax = plt.subplots(figsize=(9.5, 5.8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.8, edge_color="#435466")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2300, node_color="#eaf2f8", edgecolors="#1f567d", linewidths=1.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5, font_weight="bold")
ax.set_title('Proof dependency map: Gromov-Witten Invariants')
ax.set_axis_off()
graph_path = save_matplotlib(fig, UNIT, "figures", "proof-dependency-map.png")
plt.close(fig)

graph_check = {
    "nodes": len(CONCEPT_NODES),
    "edges": len(CONCEPT_EDGES),
    "is_directed_acyclic_graph": nx.is_directed_acyclic_graph(G),
    "source_span": '201-256',
    "passed": len(CONCEPT_NODES) >= 5 and nx.is_directed_acyclic_graph(G),
}
graph_json = save_json(graph_check, UNIT, "checks", "proof-dependency-map.json")
display_artifact(graph_path, width=780)
graph_check


## Executable Model

This section builds a small model for one core mechanism in Gromov-Witten Invariants. The model is intentionally finite and inspectable: it creates an artifact, records a JSON check, and gives a learner a parameter to perturb in the applied lab.


In [ ]:
degrees = np.array([1, 2, 3, 4])
point_constraints = 3 * degrees - 1
expected_dim_after_constraints = (6 * degrees - 2) - 2 * point_constraints
counts = np.array([1, 1, 12, 620])

fig, ax1 = plt.subplots(figsize=(7.4, 4.8))
ax1.plot(degrees, point_constraints, marker="o", color="#1f77b4", label="point constraints")
ax1.set_xlabel("degree d")
ax1.set_ylabel("generic points", color="#1f77b4")
ax1.tick_params(axis="y", labelcolor="#1f77b4")
ax2 = ax1.twinx()
ax2.semilogy(degrees, counts, marker="s", color="#d62728", label="low-degree CP2 counts")
ax2.set_ylabel("count (log scale)", color="#d62728")
ax2.tick_params(axis="y", labelcolor="#d62728")
ax1.set_title("Dimension-zero CP2 rational-curve counts")
ax1.grid(True, alpha=0.25)
fig_path = save_matplotlib(fig, UNIT, "figures", "gromov-witten-cp2-count-ledger.png")
plt.close(fig)

check = {
    "degrees": degrees.tolist(),
    "point_constraints": point_constraints.tolist(),
    "expected_dim_after_constraints": expected_dim_after_constraints.tolist(),
    "counts": counts.tolist(),
    "passed": bool(np.all(expected_dim_after_constraints == 0) and np.all(counts > 0)),
}
check_path = save_json(check, UNIT, "checks", "gromov-witten-count-checks.json")
display_artifact(fig_path, width=720)
check


## Invariant Ledger

The ledger records the chapter vocabulary as computational objects plus explicit checks. It is a small source map inside the notebook: every row names what should be inspected when the figure or experiment is changed.


In [ ]:
ledger_rows = [{'item': 'GW invariant', 'computational_object': 'signed count of constrained curves', 'check': 'dimension zero'}, {'item': 'incidence constraint', 'computational_object': 'cycle pulled back by evaluation', 'check': 'codimension match'}, {'item': 'projective-space example', 'computational_object': 'degree d rational curves', 'check': '3d-1 point count'}, {'item': 'axiom', 'computational_object': 'identity among counts', 'check': 'consistency check'}, {'item': 'deformation invariance', 'computational_object': 'pseudocycle cobordism', 'check': 'unchanged count'}]
table_path = TABLE_DIR / "invariant-ledger.csv"
with table_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["item", "computational_object", "check"])
    writer.writeheader()
    writer.writerows(ledger_rows)

ledger_check = {
    "row_count": len(ledger_rows),
    "items": [row["item"] for row in ledger_rows],
    "has_source_specific_checks": all(row["check"] for row in ledger_rows),
    "passed": len(ledger_rows) >= 5 and all(row["check"] for row in ledger_rows),
}
ledger_json = save_json(ledger_check, UNIT, "checks", "invariant-ledger.json")
display_artifact(table_path)
ledger_check


## Applied Lab

Change the number of point constraints for one degree. The expected dimension will cease to be zero, and the entry should be interpreted as a family or an overconstrained problem rather than a Gromov-Witten count.

The intended workflow is to change one parameter, rerun the executable model, and then inspect both the figure and JSON check. If the visual impression and the invariant check disagree, trust the check first and then ask what the visualization is hiding.


## Takeaways

    - A Gromov-Witten invariant is a constrained, oriented, deformation-stable count.
- Dimension matching explains why the same moduli space can produce numbers or families.
- Projective-space examples make the abstract evaluation-map construction concrete.

    ## Sanity Checks

    The final cell asserts that the generated figures, ledgers, and JSON checks exist, are nonempty, and report successful chapter-specific invariants.


In [ ]:
expected = [
    FIG_DIR / "proof-dependency-map.png",
    FIG_DIR / 'gromov-witten-cp2-count-ledger.png',
    CHECK_DIR / "proof-dependency-map.json",
    CHECK_DIR / 'gromov-witten-count-checks.json',
    CHECK_DIR / "invariant-ledger.json",
    TABLE_DIR / "invariant-ledger.csv",
]
for path in expected:
    min_bytes = 80 if path.suffix == ".csv" else 512
    assert_artifact(path, min_bytes=min_bytes)

for path in [CHECK_DIR / "proof-dependency-map.json", CHECK_DIR / 'gromov-witten-count-checks.json', CHECK_DIR / "invariant-ledger.json"]:
    data = json.loads(path.read_text(encoding="utf-8"))
    assert data.get("passed") is True, path

print(f"Validated {len(expected)} artifacts for {UNIT}")
